# Final all-task SSL comparison, ps32

This notebook creates the final publication-quality comparison of the ps32 ResNet-18 models evaluated under identical 5-cluster spatial cross-validation. It reads only existing fold metrics and prediction files, then writes final summary tables and figures. No model training, fine-tuning, checkpoint modification, or susceptibility mapping is performed here.

## 1. Purpose and experiment configuration

The comparison includes Scratch, Masked reconstruction, Contrastive learning, Jigsaw, and Rotation prediction. The primary evaluation remains fold-wise spatial cross-validation metrics; pooled ROC and precision-recall curves are included only for visualization.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PATCH_SIZE = 32
FOLDS = [0, 1, 2, 3, 4]

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "summary_all"
FIGURE_DIR = PROJECT_ROOT / "figures" / "R1_ssl_task_comparison" / "summary_all"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FOLD_METRICS_OUT = OUTPUT_DIR / "all_ssl_tasks_ps32_fold_metrics.csv"
SUMMARY_METRICS_OUT = OUTPUT_DIR / "all_ssl_tasks_ps32_summary_metrics.csv"
TRANSFER_SUMMARY_OUT = OUTPUT_DIR / "pretext_transfer_summary_ps32.csv"
MANUSCRIPT_TABLE_OUT = OUTPUT_DIR / "Table_R1_all_ssl_tasks_ps32_manuscript.csv"

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)
print("Figure directory:", FIGURE_DIR)

Project root: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project
Output directory: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all
Figure directory: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary_all


## 2. Import packages and project modules

In [2]:
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.metrics import safe_average_precision, safe_roc_auc
from src.plotting import set_publication_plot_style

pd.set_option("display.max_columns", 200)

## 3. Set paths and publication-quality plotting style

In [3]:
applied_font = set_publication_plot_style(font_family="Arial", font_size=10)
print("Applied plotting font:", applied_font)

MODEL_CONFIGS = [
    {
        "model_display_name": "Scratch",
        "model_key": "scratch_resnet18",
        "pretraining": "none",
        "pretext_category": "None",
        "fold_metrics": PROJECT_ROOT / "outputs/R0_resnet18_scratch_baseline/metrics/scratch_resnet18_ps32_fold_metrics.csv",
        "summary_metrics": PROJECT_ROOT / "outputs/R0_resnet18_scratch_baseline/metrics/scratch_resnet18_ps32_summary_metrics.csv",
        "prediction_template": PROJECT_ROOT / "outputs/R0_resnet18_scratch_baseline/predictions/scratch_resnet18_ps32_fold{fold}_predictions.csv",
    },
    {
        "model_display_name": "Masked reconstruction",
        "model_key": "masked_recon_resnet18",
        "pretraining": "masked_recon",
        "pretext_category": "Reconstruction-based",
        "fold_metrics": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/masked_recon/metrics/masked_recon_resnet18_ps32_fold_metrics.csv",
        "summary_metrics": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/masked_recon/metrics/masked_recon_resnet18_ps32_summary_metrics.csv",
        "prediction_template": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/masked_recon/predictions/masked_recon_resnet18_ps32_fold{fold}_predictions.csv",
    },
    {
        "model_display_name": "Contrastive learning",
        "model_key": "contrastive_resnet18",
        "pretraining": "contrastive",
        "pretext_category": "Contrastive",
        "fold_metrics": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/contrastive/metrics/contrastive_resnet18_ps32_fold_metrics.csv",
        "summary_metrics": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/contrastive/metrics/contrastive_resnet18_ps32_summary_metrics.csv",
        "prediction_template": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/contrastive/predictions/contrastive_resnet18_ps32_fold{fold}_predictions.csv",
    },
    {
        "model_display_name": "Jigsaw",
        "model_key": "jigsaw_resnet18",
        "pretraining": "jigsaw",
        "pretext_category": "Spatial-ordering",
        "fold_metrics": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/jigsaw/metrics/jigsaw_resnet18_ps32_fold_metrics.csv",
        "summary_metrics": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/jigsaw/metrics/jigsaw_resnet18_ps32_summary_metrics.csv",
        "prediction_template": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/jigsaw/predictions/jigsaw_resnet18_ps32_fold{fold}_predictions.csv",
    },
    {
        "model_display_name": "Rotation prediction",
        "model_key": "rotation_resnet18",
        "pretraining": "rotation",
        "pretext_category": "Geometric",
        "fold_metrics": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/rotation/metrics/rotation_resnet18_ps32_fold_metrics.csv",
        "summary_metrics": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/rotation/metrics/rotation_resnet18_ps32_summary_metrics.csv",
        "prediction_template": PROJECT_ROOT / "outputs/R1_ssl_task_comparison/rotation/predictions/rotation_resnet18_ps32_fold{fold}_predictions.csv",
    },
]

MODEL_ORDER = [config["model_display_name"] for config in MODEL_CONFIGS]
PALETTE = {
    "Scratch": "#4D4D4D",
    "Masked reconstruction": "#0072B2",
    "Contrastive learning": "#D55E00",
    "Jigsaw": "#009E73",
    "Rotation prediction": "#CC79A7",
}
print("Model order:", MODEL_ORDER)

Applied plotting font: Arial
Model order: ['Scratch', 'Masked reconstruction', 'Contrastive learning', 'Jigsaw', 'Rotation prediction']


## 4. Load fold metrics for all five models

In [4]:
fold_metric_frames = []
for config in MODEL_CONFIGS:
    path = config["fold_metrics"]
    if not path.exists():
        raise FileNotFoundError(f"Missing fold metrics file: {path}")
    df = pd.read_csv(path)
    df["model_display_name"] = config["model_display_name"]
    df["model_key"] = config["model_key"]
    df["pretraining"] = config["pretraining"]
    df["pretext_category"] = config["pretext_category"]
    fold_metric_frames.append(df)

combined_fold_metrics = pd.concat(fold_metric_frames, ignore_index=True)
combined_fold_metrics["patch_size"] = combined_fold_metrics["patch_size"].astype(int)
combined_fold_metrics.groupby("model_display_name")[["auc", "pr_auc"]].agg(["mean", "std"])

auc              pr_auc          
                           mean       std      mean       std
model_display_name                                           
Contrastive learning   0.838008  0.084941  0.831146  0.071061
Jigsaw                 0.722506  0.105176  0.715090  0.091449
Masked reconstruction  0.820912  0.043011  0.813848  0.064984
Rotation prediction    0.725500  0.091522  0.701911  0.075666
Scratch                0.778842  0.104754  0.770678  0.114186

## 5. Load fold prediction files for all five models

In [5]:
prediction_frames = []
missing_prediction_files = []
for config in MODEL_CONFIGS:
    for fold in FOLDS:
        path = Path(str(config["prediction_template"]).format(fold=fold))
        if not path.exists():
            missing_prediction_files.append(str(path))
            continue
        pred = pd.read_csv(path)
        pred["model_display_name"] = config["model_display_name"]
        pred["model_key"] = config["model_key"]
        pred["pretraining"] = config["pretraining"]
        pred["pretext_category"] = config["pretext_category"]
        prediction_frames.append(pred)

if missing_prediction_files:
    raise FileNotFoundError("Missing prediction files:\n" + "\n".join(missing_prediction_files))

combined_predictions = pd.concat(prediction_frames, ignore_index=True)
print("Loaded prediction rows:", len(combined_predictions))
combined_predictions.groupby(["model_display_name", "fold"]).size().unstack()

Loaded prediction rows: 1720


fold,0,1,2,3,4
model_display_name,,,,,
Contrastive learning,64,78,106,28,68
Jigsaw,64,78,106,28,68
Masked reconstruction,64,78,106,28,68
Rotation prediction,64,78,106,28,68
Scratch,64,78,106,28,68


## 6. Validate metrics and predictions

In [6]:
required_metric_columns = [
    "model_display_name", "model_key", "pretraining", "pretext_category", "patch_size", "fold",
    "n_train", "n_val", "n_test", "n_test_pos", "n_test_neg", "auc", "pr_auc", "accuracy_05",
    "precision_05", "recall_05", "f1_05", "best_threshold_f1", "accuracy_best_f1",
    "precision_best_f1", "recall_best_f1", "f1_best_f1", "tn", "fp", "fn", "tp",
    "best_epoch", "best_val_auc", "best_val_loss",
]
required_prediction_columns = [
    "sample_id", "x", "y", "label", "source", "cluster_id", "fold", "y_true", "y_logit",
    "y_prob", "y_pred_05", "y_pred_best_f1", "split",
]

missing_metric_cols = [col for col in required_metric_columns if col not in combined_fold_metrics.columns]
missing_prediction_cols = [col for col in required_prediction_columns if col not in combined_predictions.columns]
if missing_metric_cols:
    raise ValueError(f"Missing required metric columns: {missing_metric_cols}")
if missing_prediction_cols:
    raise ValueError(f"Missing required prediction columns: {missing_prediction_cols}")

warnings_list = []
for config in MODEL_CONFIGS:
    name = config["model_display_name"]
    metric_folds = sorted(combined_fold_metrics.loc[combined_fold_metrics["model_display_name"] == name, "fold"].astype(int).unique().tolist())
    pred_folds = sorted(combined_predictions.loc[combined_predictions["model_display_name"] == name, "fold"].astype(int).unique().tolist())
    if metric_folds != FOLDS:
        warnings_list.append(f"{name}: metric folds are {metric_folds}, expected {FOLDS}.")
    if pred_folds != FOLDS:
        warnings_list.append(f"{name}: prediction folds are {pred_folds}, expected {FOLDS}.")

if not combined_predictions["y_prob"].between(0, 1).all():
    warnings_list.append("Some predicted probabilities are outside [0, 1].")
if not set(combined_predictions["y_true"].dropna().astype(int).unique()).issubset({0, 1}):
    warnings_list.append("Prediction y_true values are not binary.")

for (model, fold), group in combined_predictions.groupby(["model_display_name", "fold"]):
    if group["sample_id"].duplicated().any():
        warnings_list.append(f"{model}, fold {fold}: duplicate sample IDs in predictions.")

for fold in FOLDS:
    id_sets = {}
    n_pos_neg = {}
    for config in MODEL_CONFIGS:
        name = config["model_display_name"]
        group = combined_predictions[(combined_predictions["model_display_name"] == name) & (combined_predictions["fold"] == fold)]
        id_sets[name] = set(group["sample_id"].astype(str))
        n_pos_neg[name] = (int((group["y_true"] == 1).sum()), int((group["y_true"] == 0).sum()))
    reference_ids = id_sets[MODEL_ORDER[0]]
    reference_counts = n_pos_neg[MODEL_ORDER[0]]
    for name in MODEL_ORDER:
        if id_sets[name] != reference_ids:
            warnings_list.append(f"Fold {fold}: test sample IDs differ for {name}.")
        if n_pos_neg[name] != reference_counts:
            warnings_list.append(f"Fold {fold}: test label counts differ for {name}: {n_pos_neg[name]} vs {reference_counts}.")

if warnings_list:
    print("Validation warnings:")
    for warning in warnings_list:
        print("WARNING:", warning)
else:
    print("Validation passed without warnings.")

Validation passed without warnings.


## 7. Create combined fold metrics table

In [7]:
combined_fold_metrics_out = combined_fold_metrics[required_metric_columns].copy()
combined_fold_metrics_out["model_display_name"] = pd.Categorical(
    combined_fold_metrics_out["model_display_name"], categories=MODEL_ORDER, ordered=True
)
combined_fold_metrics_out = combined_fold_metrics_out.sort_values(["model_display_name", "fold"]).reset_index(drop=True)
combined_fold_metrics_out.to_csv(FOLD_METRICS_OUT, index=False)
print("Saved combined fold metrics:", FOLD_METRICS_OUT)
combined_fold_metrics_out

Saved combined fold metrics: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all\all_ssl_tasks_ps32_fold_metrics.csv


,model_display_name,model_key,pretraining,pretext_category,patch_size,fold,n_train,n_val,n_test,n_test_pos,n_test_neg,auc,pr_auc,accuracy_05,precision_05,recall_05,f1_05,best_threshold_f1,accuracy_best_f1,precision_best_f1,recall_best_f1,f1_best_f1,tn,fp,fn,tp,best_epoch,best_val_auc,best_val_loss
0,Scratch,scratch_resnet18,none,None,32,0,224,56,64,32,32,0.624023,0.635310,0.562500,0.562500,0.562500,0.562500,0.219032,0.562500,0.540000,0.843750,0.658537,9,23,5,27,2,0.853316,0.972595
1,Scratch,scratch_resnet18,none,None,32,1,212,54,78,39,39,0.769231,0.788888,0.653846,0.875000,0.358974,0.509091,0.910868,0.602564,0.900000,0.230769,0.367347,38,1,30,9,54,0.947874,0.417698
2,Scratch,scratch_resnet18,none,None,32,2,190,48,106,53,53,0.754361,0.677231,0.679245,0.620253,0.924528,0.742424,0.015511,0.566038,0.536082,0.981132,0.693333,8,45,1,52,34,0.880208,0.644889
3,Scratch,scratch_resnet18,none,None,32,3,252,64,28,14,14,0.846939,0.840001,0.785714,0.722222,0.928571,0.812500,0.444866,0.785714,0.722222,0.928571,0.812500,9,5,1,13,16,0.963867,0.248575
4,Scratch,scratch_resnet18,none,None,32,4,220,56,68,34,34,0.899654,0.911957,0.764706,1.000000,0.529412,0.692308,0.152266,0.779412,0.851852,0.676471,0.754098,30,4,11,23,12,0.920918,0.439098
5,Masked reconstruction,masked_recon_resnet18,masked_recon,Reconstruction-based,32,0,224,56,64,32,32,0.895508,0.877846,0.687500,0.620000,0.968750,0.756098,0.293102,0.609375,0.561404,1.000000,0.719101,7,25,0,32,10,0.873724,0.473796
6,Masked reconstruction,masked_recon_resnet18,masked_recon,Reconstruction-based,32,1,212,54,78,39,39,0.786325,0.795165,0.653846,0.833333,0.384615,0.526316,0.615140,0.589744,0.818182,0.230769,0.360000,37,2,30,9,8,0.913580,0.445695
7,Masked reconstruction,masked_recon_resnet18,masked_recon,Reconstruction-based,32,2,190,48,106,53,53,0.799217,0.710585,0.632075,0.576087,1.000000,0.731034,0.557602,0.707547,0.648649,0.905660,0.755906,27,26,5,48,4,0.888889,0.566667
8,Masked reconstruction,masked_recon_resnet18,masked_recon,Reconstruction-based,32,3,252,64,28,14,14,0.811224,0.850065,0.785714,0.785714,0.785714,0.785714,0.519105,0.785714,0.785714,0.785714,0.785714,11,3,3,11,31,0.942383,0.307349
9,Masked reconstruction,masked_recon_resnet18,masked_recon,Reconstruction-based,32,4,220,56,68,34,34,0.812284,0.835577,0.779412,0.851852,0.676471,0.754098,0.276188,0.691176,0.666667,0.764706,0.712329,21,13,8,26,17,0.909439,0.391490


## 8. Create final summary metrics table

In [8]:
summary_rows = []
for config in MODEL_CONFIGS:
    name = config["model_display_name"]
    df = combined_fold_metrics[combined_fold_metrics["model_display_name"] == name].copy()
    summary_rows.append({
        "model_display_name": name,
        "model_key": config["model_key"],
        "pretraining": config["pretraining"],
        "pretext_category": config["pretext_category"],
        "patch_size": PATCH_SIZE,
        "mean_auc": df["auc"].mean(),
        "std_auc": df["auc"].std(ddof=1),
        "median_auc": df["auc"].median(),
        "min_auc": df["auc"].min(),
        "max_auc": df["auc"].max(),
        "worst_fold_auc": df["auc"].min(),
        "best_fold_auc": df["auc"].max(),
        "mean_pr_auc": df["pr_auc"].mean(),
        "std_pr_auc": df["pr_auc"].std(ddof=1),
        "mean_f1_05": df["f1_05"].mean(),
        "std_f1_05": df["f1_05"].std(ddof=1),
        "mean_recall_05": df["recall_05"].mean(),
        "std_recall_05": df["recall_05"].std(ddof=1),
        "mean_precision_05": df["precision_05"].mean(),
        "std_precision_05": df["precision_05"].std(ddof=1),
        "mean_accuracy_05": df["accuracy_05"].mean(),
        "std_accuracy_05": df["accuracy_05"].std(ddof=1),
        "mean_f1_best": df["f1_best_f1"].mean(),
        "std_f1_best": df["f1_best_f1"].std(ddof=1),
    })

summary_metrics = pd.DataFrame(summary_rows)
scratch_row = summary_metrics.loc[summary_metrics["model_display_name"] == "Scratch"].iloc[0]
summary_metrics["delta_mean_auc_vs_scratch"] = summary_metrics["mean_auc"] - scratch_row["mean_auc"]
summary_metrics["delta_std_auc_vs_scratch"] = summary_metrics["std_auc"] - scratch_row["std_auc"]
summary_metrics["delta_worst_auc_vs_scratch"] = summary_metrics["worst_fold_auc"] - scratch_row["worst_fold_auc"]
summary_metrics["rank_by_mean_auc"] = summary_metrics["mean_auc"].rank(ascending=False, method="min").astype(int)
summary_metrics["rank_by_stability"] = summary_metrics["std_auc"].rank(ascending=True, method="min").astype(int)
summary_metrics["rank_by_worst_fold_auc"] = summary_metrics["worst_fold_auc"].rank(ascending=False, method="min").astype(int)

role_map = {
    "Scratch": "Supervised baseline; high spatial fold variability.",
    "Masked reconstruction": "Most stable SSL transfer; best worst-fold AUC.",
    "Contrastive learning": "Highest mean AUC; effective but less stable than masked reconstruction.",
    "Jigsaw": "Weak SSL transfer; underperforms Scratch.",
    "Rotation prediction": "Pretext task solved, but downstream transfer is weak.",
}
summary_metrics["main_scientific_role"] = summary_metrics["model_display_name"].map(role_map)

summary_columns = [
    "model_display_name", "model_key", "pretraining", "pretext_category", "patch_size", "mean_auc", "std_auc",
    "median_auc", "min_auc", "max_auc", "worst_fold_auc", "best_fold_auc", "mean_pr_auc", "std_pr_auc",
    "mean_f1_05", "std_f1_05", "mean_recall_05", "std_recall_05", "mean_precision_05", "std_precision_05",
    "mean_accuracy_05", "std_accuracy_05", "mean_f1_best", "std_f1_best", "delta_mean_auc_vs_scratch",
    "delta_std_auc_vs_scratch", "delta_worst_auc_vs_scratch", "rank_by_mean_auc", "rank_by_stability",
    "rank_by_worst_fold_auc", "main_scientific_role",
]
summary_metrics = summary_metrics[summary_columns]
summary_metrics["model_display_name"] = pd.Categorical(summary_metrics["model_display_name"], categories=MODEL_ORDER, ordered=True)
summary_metrics = summary_metrics.sort_values("model_display_name").reset_index(drop=True)
summary_metrics.to_csv(SUMMARY_METRICS_OUT, index=False)
print("Saved summary metrics:", SUMMARY_METRICS_OUT)
summary_metrics

Saved summary metrics: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all\all_ssl_tasks_ps32_summary_metrics.csv


,model_display_name,model_key,pretraining,pretext_category,patch_size,mean_auc,std_auc,median_auc,min_auc,max_auc,worst_fold_auc,best_fold_auc,mean_pr_auc,std_pr_auc,mean_f1_05,std_f1_05,mean_recall_05,std_recall_05,mean_precision_05,std_precision_05,mean_accuracy_05,std_accuracy_05,mean_f1_best,std_f1_best,delta_mean_auc_vs_scratch,delta_std_auc_vs_scratch,delta_worst_auc_vs_scratch,rank_by_mean_auc,rank_by_stability,rank_by_worst_fold_auc,main_scientific_role
0,Scratch,scratch_resnet18,none,None,32,0.778842,0.104754,0.769231,0.624023,0.899654,0.624023,0.899654,0.770678,0.114186,0.663765,0.125800,0.660797,0.254597,0.755995,0.180767,0.689202,0.090025,0.657163,0.172358,0.000000,0.000000,0.000000,3,4,3,Supervised baseline; high spatial fold variabi...
1,Masked reconstruction,masked_recon_resnet18,masked_recon,Reconstruction-based,32,0.820912,0.043011,0.811224,0.786325,0.895508,0.786325,0.895508,0.813848,0.064984,0.710652,0.104860,0.763110,0.249840,0.733397,0.126848,0.707710,0.071162,0.666610,0.173936,0.042070,-0.061742,0.162301,2,1,1,Most stable SSL transfer; best worst-fold AUC.
2,Contrastive learning,contrastive_resnet18,contrastive,Contrastive,32,0.838008,0.084941,0.872070,0.704142,0.910644,0.704142,0.910644,0.831146,0.071061,0.725748,0.115135,0.722677,0.234917,0.790362,0.097063,0.748769,0.062558,0.674578,0.171529,0.059166,-0.019812,0.080119,1,2,2,Highest mean AUC; effective but less stable th...
3,Jigsaw,jigsaw_resnet18,jigsaw,Spatial-ordering,32,0.722506,0.105176,0.690335,0.605536,0.843750,0.605536,0.843750,0.715090,0.091449,0.640955,0.184169,0.652255,0.278139,0.676622,0.117925,0.669974,0.128138,0.634114,0.149763,-0.056336,0.000422,-0.018487,5,5,4,Weak SSL transfer; underperforms Scratch.
4,Rotation prediction,rotation_resnet18,rotation,Geometric,32,0.725500,0.091522,0.724457,0.602041,0.856445,0.602041,0.856445,0.701911,0.075666,0.585661,0.159821,0.579689,0.257090,0.691433,0.145365,0.630946,0.091704,0.633593,0.196475,-0.053342,-0.013231,-0.021983,4,3,5,"Pretext task solved, but downstream transfer i..."


## 9. Create pretext-task transfer summary table

In [9]:
transfer_rows = [
    ["Scratch", "None", "Not applicable", "Reference baseline", "Supervised baseline; no SSL pretraining."],
    ["Masked reconstruction", "Masked reconstruction", "Solved sufficiently for representation learning", "Effective and stable", "Effective SSL; strongest spatial stability and best worst-fold AUC."],
    ["Contrastive learning", "SimCLR-style contrastive learning", "Solved sufficiently for representation learning", "Effective average discrimination", "Effective SSL; highest mean AUC."],
    ["Jigsaw", "Jigsaw permutation prediction", "Near random", "Weak and below Scratch", "Weak or ineffective SSL; pretext task near random and downstream underperforms Scratch."],
    ["Rotation prediction", "Rotation prediction", "Solved almost perfectly", "Weak and below Scratch", "Pretext task solved almost perfectly, but downstream transfer is weak and underperforms Scratch."],
]
transfer_summary = pd.DataFrame(
    transfer_rows,
    columns=["model_display_name", "pretext_task", "pretext_task_outcome", "downstream_transfer_outcome", "scientific_interpretation"],
).merge(summary_metrics[["model_display_name", "mean_auc", "std_auc", "worst_fold_auc"]], on="model_display_name", how="left")
transfer_summary = transfer_summary[
    ["model_display_name", "pretext_task", "pretext_task_outcome", "downstream_transfer_outcome", "mean_auc", "std_auc", "worst_fold_auc", "scientific_interpretation"]
]
transfer_summary.to_csv(TRANSFER_SUMMARY_OUT, index=False)
print("Saved pretext transfer summary:", TRANSFER_SUMMARY_OUT)
transfer_summary

Saved pretext transfer summary: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all\pretext_transfer_summary_ps32.csv


,model_display_name,pretext_task,pretext_task_outcome,downstream_transfer_outcome,mean_auc,std_auc,worst_fold_auc,scientific_interpretation
0,Scratch,None,Not applicable,Reference baseline,0.778842,0.104754,0.624023,Supervised baseline; no SSL pretraining.
1,Masked reconstruction,Masked reconstruction,Solved sufficiently for representation learning,Effective and stable,0.820912,0.043011,0.786325,Effective SSL; strongest spatial stability and...
2,Contrastive learning,SimCLR-style contrastive learning,Solved sufficiently for representation learning,Effective average discrimination,0.838008,0.084941,0.704142,Effective SSL; highest mean AUC.
3,Jigsaw,Jigsaw permutation prediction,Near random,Weak and below Scratch,0.722506,0.105176,0.605536,Weak or ineffective SSL; pretext task near ran...
4,Rotation prediction,Rotation prediction,Solved almost perfectly,Weak and below Scratch,0.725500,0.091522,0.602041,"Pretext task solved almost perfectly, but down..."


## 10. Create manuscript-ready result table

In [10]:
finding_map = {
    "Scratch": "Baseline; high spatial fold variability",
    "Masked reconstruction": "Most stable transfer and best worst-fold AUC",
    "Contrastive learning": "Highest mean AUC",
    "Jigsaw": "Weak transfer; below Scratch",
    "Rotation prediction": "Pretext solved, but weak downstream transfer",
}
pretext_display = {
    "Scratch": "None",
    "Masked reconstruction": "Masked reconstruction",
    "Contrastive learning": "Contrastive learning",
    "Jigsaw": "Jigsaw",
    "Rotation prediction": "Rotation prediction",
}
manuscript_table = pd.DataFrame({
    "Model": summary_metrics["model_display_name"].astype(str),
    "Pretext task": summary_metrics["model_display_name"].astype(str).map(pretext_display),
    "Mean AUC": summary_metrics["mean_auc"].round(3),
    "AUC SD": summary_metrics["std_auc"].round(3),
    "Worst-fold AUC": summary_metrics["worst_fold_auc"].round(3),
    "Mean PR-AUC": summary_metrics["mean_pr_auc"].round(3),
    "Mean F1 at 0.5": summary_metrics["mean_f1_05"].round(3),
    "Main finding": summary_metrics["model_display_name"].astype(str).map(finding_map),
})
manuscript_table.to_csv(MANUSCRIPT_TABLE_OUT, index=False)
print("Saved manuscript-ready table:", MANUSCRIPT_TABLE_OUT)
manuscript_table

Saved manuscript-ready table: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all\Table_R1_all_ssl_tasks_ps32_manuscript.csv


,Model,Pretext task,Mean AUC,AUC SD,Worst-fold AUC,Mean PR-AUC,Mean F1 at 0.5,Main finding
0,Scratch,None,0.779,0.105,0.624,0.771,0.664,Baseline; high spatial fold variability
1,Masked reconstruction,Masked reconstruction,0.821,0.043,0.786,0.814,0.711,Most stable transfer and best worst-fold AUC
2,Contrastive learning,Contrastive learning,0.838,0.085,0.704,0.831,0.726,Highest mean AUC
3,Jigsaw,Jigsaw,0.723,0.105,0.606,0.715,0.641,Weak transfer; below Scratch
4,Rotation prediction,Rotation prediction,0.726,0.092,0.602,0.702,0.586,"Pretext solved, but weak downstream transfer"


## 11. Generate final publication-quality figures

Pooled ROC and precision-recall curves are shown only for visualization. Primary evaluation is based on fold-wise SCV metrics.

In [11]:
def save_figure(fig, stem: str):
    png_path = FIGURE_DIR / f"{stem}.png"
    pdf_path = FIGURE_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=600, bbox_inches="tight", transparent=False, facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", transparent=False, facecolor="white")
    plt.close(fig)
    return png_path, pdf_path


def clean_axes(ax, keep_grid=False):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if keep_grid:
        ax.grid(axis="y", color="0.88", linewidth=0.6)
    return ax


def roc_curve_points(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    thresholds = np.r_[np.inf, np.sort(np.unique(y_prob))[::-1], -np.inf]
    positives = max(int((y_true == 1).sum()), 1)
    negatives = max(int((y_true == 0).sum()), 1)
    fpr, tpr = [], []
    for threshold in thresholds:
        pred = y_prob >= threshold
        tp = int(((y_true == 1) & pred).sum())
        fp = int(((y_true == 0) & pred).sum())
        tpr.append(tp / positives)
        fpr.append(fp / negatives)
    return np.asarray(fpr), np.asarray(tpr)


def pr_curve_points(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    thresholds = np.r_[np.inf, np.sort(np.unique(y_prob))[::-1], -np.inf]
    positives = max(int((y_true == 1).sum()), 1)
    precision, recall = [], []
    for threshold in thresholds:
        pred = y_prob >= threshold
        tp = int(((y_true == 1) & pred).sum())
        fp = int(((y_true == 0) & pred).sum())
        precision.append(tp / (tp + fp) if (tp + fp) else 1.0)
        recall.append(tp / positives)
    return np.asarray(recall), np.asarray(precision)


summary_ordered = summary_metrics.copy()
summary_ordered["model_display_name"] = pd.Categorical(summary_ordered["model_display_name"], categories=MODEL_ORDER, ordered=True)
summary_ordered = summary_ordered.sort_values("model_display_name").reset_index(drop=True)
figure_paths = []
x = np.arange(len(MODEL_ORDER))
means = summary_ordered["mean_auc"].to_numpy()
stds = summary_ordered["std_auc"].to_numpy()
colors = [PALETTE[name] for name in MODEL_ORDER]

In [12]:
# Figure R1a: mean AUC with fold variability.
fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.bar(x, means, yerr=stds, color=colors, edgecolor="black", linewidth=0.6, capsize=3, alpha=0.9)
for index, name in enumerate(MODEL_ORDER):
    fold_vals = combined_fold_metrics.loc[combined_fold_metrics["model_display_name"] == name, "auc"].to_numpy()
    jitter = np.linspace(-0.13, 0.13, len(fold_vals))
    ax.scatter(np.full(len(fold_vals), x[index]) + jitter, fold_vals, s=18, color="white", edgecolor="black", linewidth=0.5, zorder=4)
    if name != "Scratch":
        delta = summary_ordered.loc[summary_ordered["model_display_name"] == name, "delta_mean_auc_vs_scratch"].iloc[0]
        ax.text(x[index], means[index] + stds[index] + 0.025, f"{delta:+.3f}", ha="center", va="bottom")
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, rotation=20, ha="right")
ax.set_ylabel("Mean AUC")
ax.set_ylim(0.45, 1.0)
ax.set_title("Downstream performance across SSL pretext tasks")
clean_axes(ax, keep_grid=True)
fig.tight_layout()
figure_paths.extend(save_figure(fig, "Fig_R1a_all_tasks_mean_auc_ps32"))

In [13]:
# Figure R1b: fold-wise AUC comparison.
fig, ax = plt.subplots(figsize=(7.2, 4.5))
for name in MODEL_ORDER:
    df = combined_fold_metrics[combined_fold_metrics["model_display_name"] == name].sort_values("fold")
    ax.plot(df["fold"], df["auc"], marker="o", color=PALETTE[name], label=name)
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.9)
ax.set_xticks(FOLDS)
ax.set_xlabel("SCV fold")
ax.set_ylabel("AUC")
ax.set_ylim(0.45, 1.0)
ax.set_title("Fold-wise spatial generalization")
ax.legend(frameon=False, loc="lower right")
clean_axes(ax, keep_grid=True)
fig.tight_layout()
figure_paths.extend(save_figure(fig, "Fig_R1b_all_tasks_foldwise_auc_ps32"))

In [14]:
# Figure R1c: accuracy-stability trade-off.
fig, ax = plt.subplots(figsize=(6.8, 4.8))
for _, row in summary_ordered.iterrows():
    name = str(row["model_display_name"])
    ax.scatter(row["std_auc"], row["mean_auc"], s=70, color=PALETTE[name], edgecolor="black", linewidth=0.6, zorder=3)
    dx = 0.004 if name != "Contrastive learning" else -0.045
    dy = 0.006 if name != "Rotation prediction" else -0.018
    ax.text(row["std_auc"] + dx, row["mean_auc"] + dy, name, ha="left" if dx > 0 else "right", va="center")
ax.set_xlabel("AUC SD")
ax.set_ylabel("Mean AUC")
ax.set_xlim(0.02, 0.12)
ax.set_ylim(0.68, 0.87)
ax.set_title("Accuracy-stability trade-off")
clean_axes(ax, keep_grid=True)
fig.tight_layout()
figure_paths.extend(save_figure(fig, "Fig_R1c_auc_stability_tradeoff_ps32"))

In [15]:
# Figure R1d: worst-fold AUC and AUC SD.
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3))
for axis, metric, ylabel, panel in [
    (axes[0], "worst_fold_auc", "Worst-fold AUC", "(a)"),
    (axes[1], "std_auc", "AUC SD", "(b)"),
]:
    values = summary_ordered[metric].to_numpy()
    axis.bar(x, values, color=colors, edgecolor="black", linewidth=0.6, alpha=0.9)
    axis.set_xticks(x)
    axis.set_xticklabels(MODEL_ORDER, rotation=25, ha="right")
    axis.set_ylabel(ylabel)
    axis.text(0.01, 0.98, panel, transform=axis.transAxes, ha="left", va="top")
    clean_axes(axis, keep_grid=True)
axes[0].axhline(0.5, color="black", linestyle="--", linewidth=0.9)
axes[0].set_ylim(0.45, 0.9)
axes[1].set_ylim(0.0, max(summary_ordered["std_auc"]) + 0.03)
fig.suptitle("Robustness under spatial cross-validation")
fig.tight_layout()
figure_paths.extend(save_figure(fig, "Fig_R1d_worstfold_stability_ps32"))

In [16]:
# Figure R1e: pooled ROC curves.
fig, ax = plt.subplots(figsize=(6, 5))
for name in MODEL_ORDER:
    pred = combined_predictions[combined_predictions["model_display_name"] == name]
    y_true = pred["y_true"].to_numpy()
    y_prob = pred["y_prob"].to_numpy()
    auc = safe_roc_auc(y_true, y_prob)
    fpr, tpr = roc_curve_points(y_true, y_prob)
    ax.plot(fpr, tpr, color=PALETTE[name], label=f"{name} AUC {auc:.3f}")
ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=0.9, label="Random baseline")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Pooled ROC curves")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(frameon=False, loc="lower right")
clean_axes(ax, keep_grid=True)
fig.tight_layout()
figure_paths.extend(save_figure(fig, "Fig_R1e_pooled_roc_all_tasks_ps32"))
print("Pooled ROC and PR curves are shown only for visualization. Primary evaluation is based on fold-wise SCV metrics.")

Pooled ROC and PR curves are shown only for visualization. Primary evaluation is based on fold-wise SCV metrics.


In [17]:
# Figure R1f: pooled precision-recall curves.
fig, ax = plt.subplots(figsize=(6, 5))
for name in MODEL_ORDER:
    pred = combined_predictions[combined_predictions["model_display_name"] == name]
    y_true = pred["y_true"].to_numpy()
    y_prob = pred["y_prob"].to_numpy()
    ap = safe_average_precision(y_true, y_prob)
    recall, precision = pr_curve_points(y_true, y_prob)
    ax.plot(recall, precision, color=PALETTE[name], label=f"{name} AP {ap:.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Pooled precision-recall curves")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(frameon=False, loc="lower left")
clean_axes(ax, keep_grid=True)
fig.tight_layout()
figure_paths.extend(save_figure(fig, "Fig_R1f_pooled_pr_all_tasks_ps32"))
print("Pooled ROC and PR curves are shown only for visualization. Primary evaluation is based on fold-wise SCV metrics.")

Pooled ROC and PR curves are shown only for visualization. Primary evaluation is based on fold-wise SCV metrics.


In [18]:
# Figure R1g: pretext-task transfer matrix.
transfer_matrix_rows = ["Masked reconstruction", "Contrastive learning", "Jigsaw", "Rotation prediction"]
transfer_matrix_columns = ["Pretext task solved", "Mean AUC gain", "Stability gain", "Worst-fold gain", "Transfer outcome"]
transfer_matrix_text = [
    ["Yes", "Positive", "Strong", "Strong", "Effective and stable"],
    ["Yes", "Strong", "Moderate", "Moderate", "Effective but less stable"],
    ["No", "Negative", "None", "Negative", "Ineffective"],
    ["Yes", "Negative", "Weak", "Negative", "Solved but weak transfer"],
]
value_color = {
    "Yes": "#D9F0D3", "No": "#F2D7D5", "Positive": "#D9F0D3", "Strong": "#A6DBA0",
    "Moderate": "#DDECCB", "Weak": "#FEE8C8", "None": "#F0F0F0", "Negative": "#F2D7D5",
    "Effective and stable": "#A6DBA0", "Effective but less stable": "#DDECCB",
    "Ineffective": "#F2D7D5", "Solved but weak transfer": "#FEE8C8",
}
fig, ax = plt.subplots(figsize=(11.6, 3.5))
ax.set_xlim(0, len(transfer_matrix_columns))
ax.set_ylim(0, len(transfer_matrix_rows))
for i, row_name in enumerate(transfer_matrix_rows):
    for j, col_name in enumerate(transfer_matrix_columns):
        y = len(transfer_matrix_rows) - i - 1
        text = transfer_matrix_text[i][j]
        ax.add_patch(plt.Rectangle((j, y), 1, 1, facecolor=value_color.get(text, "white"), edgecolor="white", linewidth=2))
        ax.text(j + 0.5, y + 0.5, text, ha="center", va="center", wrap=True)
ax.set_xticks(np.arange(len(transfer_matrix_columns)) + 0.5)
ax.set_xticklabels(transfer_matrix_columns)
ax.set_yticks(np.arange(len(transfer_matrix_rows)) + 0.5)
ax.set_yticklabels(list(reversed(transfer_matrix_rows)))
ax.set_title("Pretext-task transfer matrix")
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
figure_paths.extend(save_figure(fig, "Fig_R1g_pretext_transfer_matrix_ps32"))

In [19]:
# Figure R1h: metric heatmap.
heatmap_cols = ["Mean AUC", "AUC SD", "Worst-fold AUC", "Mean PR-AUC", "F1 at 0.5"]
heatmap_data = summary_ordered[["mean_auc", "std_auc", "worst_fold_auc", "mean_pr_auc", "mean_f1_05"]].to_numpy(dtype=float)
color_data = np.zeros_like(heatmap_data)
for j in range(heatmap_data.shape[1]):
    col = heatmap_data[:, j]
    color_data[:, j] = 0.5 if np.isclose(col.max(), col.min()) else (col - col.min()) / (col.max() - col.min())
fig, ax = plt.subplots(figsize=(7.8, 4.3))
im = ax.imshow(color_data, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(heatmap_cols)))
ax.set_xticklabels(heatmap_cols, rotation=25, ha="right")
ax.set_yticks(np.arange(len(MODEL_ORDER)))
ax.set_yticklabels(MODEL_ORDER)
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        ax.text(j, i, f"{heatmap_data[i, j]:.3f}", ha="center", va="center", color="black")
ax.set_title("Summary of downstream metrics")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cbar.ax.tick_params(labelsize=10)
cbar.set_label("Relative value", fontsize=10)
fig.tight_layout()
figure_paths.extend(save_figure(fig, "Fig_R1h_metric_heatmap_all_tasks_ps32"))

In [20]:
# Optional composite figure.
# The individual panels are clearer than a compressed composite for this result set,
# so the optional composite is intentionally skipped.
print("Optional composite figure skipped because the all-task layout is crowded; individual figures are saved instead.")

Optional composite figure skipped because the all-task layout is crowded; individual figures are saved instead.


## 12. Print key scientific findings

In [21]:
print("Final ps32 SCV summary:")
print(manuscript_table.to_string(index=False))

scratch = summary_metrics[summary_metrics["model_display_name"] == "Scratch"].iloc[0]
masked = summary_metrics[summary_metrics["model_display_name"] == "Masked reconstruction"].iloc[0]
contrastive = summary_metrics[summary_metrics["model_display_name"] == "Contrastive learning"].iloc[0]
jigsaw = summary_metrics[summary_metrics["model_display_name"] == "Jigsaw"].iloc[0]
rotation = summary_metrics[summary_metrics["model_display_name"] == "Rotation prediction"].iloc[0]

print("\nKey scientific findings:")
print("1. Masked reconstruction and contrastive learning improve mean AUC over Scratch.")
print(f"2. Contrastive learning achieves the highest mean AUC: {contrastive['mean_auc']:.3f}.")
print(f"3. Masked reconstruction achieves the lowest AUC SD ({masked['std_auc']:.3f}) and highest worst-fold AUC ({masked['worst_fold_auc']:.3f}).")
print(f"4. Jigsaw underperforms Scratch, with mean AUC {jigsaw['mean_auc']:.3f} versus Scratch {scratch['mean_auc']:.3f}.")
print(f"5. Rotation prediction solved its pretext task almost perfectly, but downstream mean AUC is {rotation['mean_auc']:.3f}, below Scratch.")
print("6. Reconstruction- and contrastive-based SSL tasks are better aligned with 14-factor geospatial raster patches than spatial-ordering or simple geometric rotation tasks.")

Final ps32 SCV summary:
                Model          Pretext task  Mean AUC  AUC SD  Worst-fold AUC  Mean PR-AUC  Mean F1 at 0.5                                 Main finding
              Scratch                  None     0.779   0.105           0.624        0.771           0.664      Baseline; high spatial fold variability
Masked reconstruction Masked reconstruction     0.821   0.043           0.786        0.814           0.711 Most stable transfer and best worst-fold AUC
 Contrastive learning  Contrastive learning     0.838   0.085           0.704        0.831           0.726                             Highest mean AUC
               Jigsaw                Jigsaw     0.723   0.105           0.606        0.715           0.641                 Weak transfer; below Scratch
  Rotation prediction   Rotation prediction     0.726   0.092           0.602        0.702           0.586 Pretext solved, but weak downstream transfer

Key scientific findings:
1. Masked reconstruction and contrasti

## 13. Save all outputs and final QA summary

In [22]:
print("Saved summary CSV files:")
for path in [FOLD_METRICS_OUT, SUMMARY_METRICS_OUT, TRANSFER_SUMMARY_OUT, MANUSCRIPT_TABLE_OUT]:
    print(path, "exists=", path.exists(), "bytes=", path.stat().st_size if path.exists() else None)

print("\nSaved figure files:")
for path in figure_paths:
    print(path, "exists=", path.exists(), "bytes=", path.stat().st_size if path.exists() else None)

figure_text_items = MODEL_ORDER + [
    "Mean AUC", "AUC SD", "Worst-fold AUC", "F1 at 0.5", "PR-AUC",
    "Downstream performance across SSL pretext tasks", "Fold-wise spatial generalization",
    "Accuracy-stability trade-off", "Robustness under spatial cross-validation",
    "Pooled ROC curves", "Pooled precision-recall curves", "Pretext-task transfer matrix",
    "Summary of downstream metrics",
]
assert all("_" not in item for item in figure_text_items)

print("\nQA summary:")
print("No training was performed.")
print("No checkpoints were modified.")
print("No existing metric or prediction files were modified.")
print("Figure text uses clean display labels without underscores.")
print("Standardized publication plotting style was applied with font:", applied_font)
print("Pooled ROC and PR curves are visualization only; fold-wise SCV metrics remain the primary evaluation.")

Saved summary CSV files:
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all\all_ssl_tasks_ps32_fold_metrics.csv exists= True bytes= 8136
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all\all_ssl_tasks_ps32_summary_metrics.csv exists= True bytes= 3099
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all\pretext_transfer_summary_ps32.csv exists= True bytes= 1213
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary_all\Table_R1_all_ssl_tasks_ps32_manuscript.csv exists= True bytes= 577

Saved figure files:
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary_all\Fig_R1a_all_tasks_mean_auc_ps32.png exists= True bytes= 415671
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary_all\Fig_R1a_all_tasks_mean